# 04 — Random Forest Model Training

In [1]:
import os
from pathlib import Path
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix)

# --- Project Path Setup ---
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)

if not os.path.exists("data/processed/machine_failure_features.csv"):
    raise FileNotFoundError(
        "Features dataset 'data/processed/machine_failure_features.csv' not found! "
        "Please run '03_feature_engineering.ipynb' first."
    )

# 1. Load the feature-engineered dataset
df = pd.read_csv("data/processed/machine_failure_features.csv")

# 2. Separate features (X) and target (y)
X = df.drop(columns=["Machine failure"])
y = df["Machine failure"]

# 3. Train/test split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("Train set failure count:", y_train.sum(), f"({y_train.mean():.4%})")
print("Test set failure count: ", y_test.sum(), f"({y_test.mean():.4%})")

# 4. Instantiate and train Random Forest Classifier
model = RandomForestClassifier(
    n_estimators=200, max_depth=20, class_weight="balanced", random_state=42
)
model.fit(X_train, y_train)

# 5. Evaluate model performance on the held-out test set
pred = model.predict(X_test)

print("\n--- Random Forest Evaluation Metrics ---")
print(f"Accuracy:  {accuracy_score(y_test, pred):.4f}")
print(f"Precision: {precision_score(y_test, pred):.4f}")
print(f"Recall:    {recall_score(y_test, pred):.4f}")
print(f"F1 Score:  {f1_score(y_test, pred):.4f}")
print("\nConfusion Matrix:\n", confusion_matrix(y_test, pred))

# 6. Feature Importance
importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\nTop 5 Most Important Features:\n", importances.head())

# 7. Create models directory and save model
os.makedirs("models", exist_ok=True)
joblib.dump(model, "models/random_forest.pkl")

# Verify the file was created
rf_saved = os.path.exists("models/random_forest.pkl")
print("\nFile exists check:", rf_saved)
if not rf_saved:
    raise FileNotFoundError("Failed to save 'models/random_forest.pkl'.")
print("Saved trained model to: models/random_forest.pkl")

Train set failure count: 271 (3.3875%)
Test set failure count:  68 (3.4000%)



--- Random Forest Evaluation Metrics ---
Accuracy:  0.9920
Precision: 0.9483
Recall:    0.8088
F1 Score:  0.8730

Confusion Matrix:
 [[1929    3]
 [  13   55]]

Top 5 Most Important Features:
 Rotational speed [rpm]         0.207225
Power [W]                      0.183197
Torque [Nm]                    0.175704
Tool wear [min]                0.148356
Wear-Torque product [minNm]    0.109950
dtype: float64

File exists check: True
Saved trained model to: models/random_forest.pkl
